$x[n]$, $h[n]$ をそれぞれ2048点の離散時間信号とし，これらの2048点巡回畳み込みを計算する．
```circular_convolve_direct(x, h, N=2048)``` を用いた場合の計算時間と，
FFT を用いて畳み込み定理に基づいて計算した場合の計算時間をそれぞれ測定し，比較しなさい．

巡回畳み込みはDFT領域で積に対応
\begin{align*}
y=x\circledast_N h
\quad\Longleftrightarrow\quad
Y[k]=X[k]H[k]
\end{align*}

直接計算はおおよそ $O(N^2)$，FFTを使う方法はおおよそ $O(N\log N)$ なので，$N=2048$ ではFFTの方がかなり高速になることが期待される

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

from time import perf_counter

def periodize(x, N):
    x = np.asarray(x)
    y = np.zeros(N, dtype=x.dtype)
    np.add.at(y, np.arange(x.size) % N, x)
    return y

def circular_convolve_direct(x, h, N=None):
    x = np.asarray(x)
    h = np.asarray(h)

    if N is None:
        if x.size != h.size:
            raise ValueError("x と h の長さが異なる場合は N を指定してください．")
        N = x.size

    xN = periodize(x, N)
    hN = periodize(h, N)

    dtype = np.result_type(xN, hN, np.float64)
    y = np.zeros(N, dtype=dtype)

    for n in range(N):
        for k in range(N):
            y[n] += xN[k] * hN[(n-k) % N]
    return y

def circular_convolve_fft(x, h, N):
    X = np.fft.fft(x, n=N)
    H = np.fft.fft(h, n=N)
    return np.fft.ifft(X * H).real

N = 2048
rng = np.random.default_rng(0)
x = rng.standard_normal(N)
h = rng.standard_normal(N)

t0 = perf_counter()
y_direct = circular_convolve_direct(x, h, N=N)
t_direct = perf_counter() - t0

t0 = perf_counter()
y_fft = circular_convolve_fft(x, h, N=N)
t_fft = perf_counter() - t0

max_error = np.max(np.abs(y_direct - y_fft))

print(f"直接計算の時間: {t_direct:.6f} s")
print(f"FFT計算の時間: {t_fft:.6f} s")
print(f"速度比 直接計算/FFT: {t_direct/t_fft:.1f}")
print(f"最大誤差: {max_error:.3e}")
print("結果が一致するか =", np.allclose(y_direct, y_fft))

直接計算の時間: 0.857616 s
FFT計算の時間: 0.001303 s
速度比 直接計算/FFT: 658.3
最大誤差: 5.400e-13
結果が一致するか = True


最大誤差が $10^{-10}$ 程度以下であれば，差は主に浮動小数点丸め誤差です．計算時間は実行環境で変わりますが，2048点ではFFTを使った方法の方が大幅に高速になります．